In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(Path.cwd())))

In [13]:
from pyiron_workflow import Workflow
from pyironflow.pyironflow import PyironFlow
from pyironflow.nodeeditor import NodeEditor
from pyiron_workflow.nodes.while_loop import while_node
from pyiron_workflow.nodes.for_loop import for_node
import pyiron_workflow as pwf
import pathlib
from pyiron_workflow import as_function_node, as_macro_node


### while-loop in the command line

To get started, we first have a look at the command line implementation of while-loops. Below we can see the basic nodes, that our loop will consist of. We will recreate Herons's Algorithm for the calculation of square roots. This algorithm repeats three simple steps. We have our number S, which we want to find the square root of, as well as a value x_n, which at the start is any number between S and 0. The formula for one step of this algorithm is x_n+1 = ((S/x_n) + x_n)/2. Below you can see the corresponding function nodes for our workflow, as well as the macro "heron_step" which represents one step of the algorithm.

In [15]:
@as_function_node()
def add(a: float | int = 0, b: float | int = 0):
    added = a+b
    return added


@as_function_node()
def divide(x: float | int = 0, y: float | int = 0):
    out = x/y
    return out

@as_function_node()
def half(x: float | int = 0):
    x_halfed = x/2
    return x_halfed

@as_macro_node()
def heron_step(self, S: float | int = 0, x_n: float | int = 0):
    self.divide = divide(S, x_n)    
    self.add = add(x_n, self.divide)
    self.half = half(self.add)

    return self.half, x_n

With each iteration, our x_n gets closer to the real square root of S. A fitting ending point would be, if our results from one step to the next only differ by a small amount. For this, we have a test node, wich compares two values and returns true, as long as the difference between the two exceeds a given value.

In [ ]:
@as_function_node()
def diff_above(x: float | int = 0, y: float | int = 0, max_diff: float | int = 0):
    
    diff = abs(x-y)
    if diff > max_diff:
        diff_above = True
    else:
        diff_above = False
        
    return diff_above

With these things combined, we can create our heron loop. 
test_node_class defines, which kind of test determines, if the loop continues.
body_node_class defines the macro, which gets executed for each step.
body_to_test connections describes what outputs of the body node get connected to the test node. Since we want to check the difference between x_n and x_n+1 we need to connect both outputs of our macro to the test nodes x and y inputs to get them tested each step.
body_to_body connections describe how the data gets passed between calculation steps. Since we want to keep our original starting number S but want to pass on our newly calculated "half", it gets connected to input "x_n".
At last we define starting values for our loop to run. After that it can be executed.

In [4]:
heron_loop = pwf.while_node(
    test_node_class = diff_above,
    body_node_class = heron_step,
    body_to_test_connections = (("half", "x"), ("x_n", "y"),),
    body_to_body_connections = ("half","x_n"),
    body_S= 100,
    body_x_n= 95,
    test_x= 1,
    test_y= 2,
    test_max_diff= 0.01,
    strict_condition_hint = False)

In [5]:
heron_loop.run()

{'half': 10.000000000035843, 'x_n': 10.00002677482755}

### while-loop in pyironFlow

The three necessary nodes are already added to the workflow. Should we wish to add more, they can be found in the loop_demo folder in the node library on the left hand side. Simply navigate through the tree and click the nodes you would like to add.
To create a while node from the three selected nodes, we must first establish the connection between them. We pick heron1 for step one of the loop and heron2 for step two. After each step, we want to check for our condition. In this case, this means comparing the value of x_n to the result of the macro computation half. for this, we connect both outputs (half, x_n) to our test node (x,y). Should the condition hold, we want to use our result "half" as the new x_n in the next loop step. To achieve this, we connect output "half" of "heron1" to input "x_n" of "heron2". Now we have made all the needed connections and we can create the loop.
For this, navigate to the dropdown menu on the left and expand the field "Loop". Now we select "while" as out loop type and enter the name "heron1" into the field "body" and "heron2" into the field "body2". Then we enter "diff" as the name of our test. We can then simply click the button below and our new loop node should appear. We can then remove the old nodes, as they are no longer needed. You can select the loop node and click "expand" to get a more detailed view of the node. Here we can switch between the "blueprint" view, which shows the dataflow connections inside of the loop, or select "control flow" which shows us the simplified control flow representation of our loop. If we want to run our loop, all we need to do is connect a few input nodes (also in the node library) and then select "pull"

In [8]:
wf_h = Workflow('heron')

wf_h.heron1 = heron_step()
wf_h.heron2 = heron_step()
wf_h.diff = diff_above()


pf = NodeEditor([wf_h])
pf.gui

### for-loop in the command line
We start again with the neccessary nodes. For this example, we have a simple algorithm. It takes a set of given numbers, adds one and then squares them. For the for loop we dont need a seperate test node. The loop ends once it has iterated over all the given values.

In [9]:
@as_function_node()
def add_1(a: float | int = 0):
    return a+1


@as_function_node()
def square(n):
    n_squared = n*n
    return n_squared

@as_macro_node()
def add_and_square_macro(self, n):

    self.add_1 = add_1(n)
    self.square = square(self.add_1)
    return self.square

body_node_class again declares, which macro shall be executed each step of the loop.
iter_on defines, which input the node iterates over
Since we dont want a dataframe as output, it gets set to false. At the end, we give a set of numbers for the input n to iterate over. After that, the loop can be executed.

In [10]:
for_add_and_square = pwf.for_node(
    body_node_class = add_and_square_macro,
    iter_on = "n",
    output_as_dataframe = False,
    n= [0,1,2,3,4],
    )

In [11]:
for_add_and_square.run()

{'n': [0, 1, 2, 3, 4], 'square': [1, 4, 9, 16, 25]}

### for-loop in pyironFlow
The necessary node is already added to the Workflow. All we need to to is got to the "Loop" dropdown again. This time, we select "for" as loop type. We declare "macro" as the node we want to loop. Then we just entwer which inputs we would like to iterate over, seperated by spaces. Since we only have the one, we enter n (without the ""). Then we can click the create button and the new loop node gets added. We have same option of expanding the loop and viewing the "blueprint", which shows the data flow connections, and "control flow" which shows the control flow representation. All we need to do now is connect an input node. There is a fitting one called "list_0_to_4" in the node library-

In [12]:
wf_ans =  Workflow('add_and_square')

wf_ans.macro = add_and_square_macro()

pf2 = NodeEditor([wf_ans])
pf2.gui